# SoftDrop splitting function

Generates QCD dijet events with **Pythia8**, clusters jets with **FastJet** anti-$k_t$,
applies **SoftDrop** grooming ($\beta=0$, $z_{\rm cut}=0.1$), and plots the
splitting-function observables $z_g$ and $\theta_g$.

**References**
- Larkoski, Marzani, Soyez, Thaler (2014) [arXiv:1402.2657](https://arxiv.org/abs/1402.2657)
- Dasgupta, Fregoso, Marzani, Salam (2013) [arXiv:1307.0004](https://arxiv.org/abs/1307.0004)

In [ ]:
# hepyy.load() is a no-op if packages were already loaded via
# `module load` or the hepyy Jupyter kernel. Safe to leave in place.
import hepyy
hepyy.load('fastjet')
hepyy.load('fjcontrib')
hepyy.load('pythia8')

In [ ]:
import cppyy
import pythia8
import fastjet
import fjcontrib

# Sanity check: cppyy location
print(f"cppyy from: {cppyy.__file__}")

PseudoJetVec = cppyy.gbl.std.vector[fastjet.PseudoJet]

## Parameters

In [ ]:
R        = 0.4
pt_min   = 100.0   # GeV — minimum jet pT
z_cut    = 0.1     # SoftDrop z_cut
beta     = 0.0     # SoftDrop beta (0 = mMDT-like)
n_events = 2000

## Jet and grooming setup

In [ ]:
jet_def = fastjet.JetDefinition(fastjet.antikt_algorithm, R)
sd      = fjcontrib.SoftDrop(beta, z_cut, R)

## Pythia8 setup

In [ ]:
pythia = pythia8.Pythia()
pythia.readString('Beams:eCM = 13000.')
pythia.readString('HardQCD:all = on')
pythia.readString('PhaseSpace:pTHatMin = 100.')
pythia.readString('Next:numberShowEvent = 0')
pythia.readString('Print:quiet = on')
pythia.init()

## Event loop

For each jet that passes SoftDrop we extract:
- $z_g = \min(p_{T1}, p_{T2}) / (p_{T1} + p_{T2})$ — momentum sharing of the first hard splitting
- $\theta_g = \Delta R_{12} / R$ — opening angle of that splitting (normalised to $R$)

In [ ]:
z_g_vals     = []
theta_g_vals = []

for _ in range(n_events):
    if not pythia.next():
        continue
    particles = PseudoJetVec()
    for i in range(pythia.event.size()):
        p = pythia.event[i]
        if p.isFinal() and p.isVisible():
            particles.push_back(fastjet.PseudoJet(p.px(), p.py(), p.pz(), p.e()))

    cs   = fastjet.ClusterSequence(particles, jet_def)
    jets = fastjet.sorted_by_pt(cs.inclusive_jets(pt_min))

    for jet in jets:
        groomed = sd.result(jet)
        if not groomed.has_pieces():
            continue
        pieces = groomed.pieces()
        if len(pieces) < 2:
            continue
        pt1, pt2 = pieces[0].pt(), pieces[1].pt()
        z_g = min(pt1, pt2) / (pt1 + pt2)
        deta = pieces[0].eta() - pieces[1].eta()
        dphi = pieces[0].delta_phi_to(pieces[1])
        theta_g = (deta**2 + dphi**2)**0.5 / R
        z_g_vals.append(z_g)
        theta_g_vals.append(theta_g)

print(f'Collected {len(z_g_vals)} groomed jets from {n_events} events')

## Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f'SoftDrop splitting function  '
    f'$\\beta={beta}$, $z_{{\\rm cut}}={z_cut}$,  '
    f'anti-$k_t$ $R={R}$,  $p_T>{pt_min}$ GeV  ({n_events} events)'
)

# z_g
ax = axes[0]
ax.hist(z_g_vals, bins=25, range=(z_cut, 0.5), histtype='step', lw=2,
        color='steelblue', density=True)
ax.axvline(z_cut, ls='--', color='gray', lw=1, label=f'$z_{{\\rm cut}}={z_cut}$')
ax.set_xlabel(r'$z_g$')
ax.set_ylabel(r'$(1/N_{\rm jet})\,dN/dz_g$')
ax.set_title(r'Momentum sharing $z_g$')
ax.legend()

# theta_g
ax = axes[1]
ax.hist(theta_g_vals, bins=25, range=(0, 1), histtype='step', lw=2,
        color='tomato', density=True)
ax.set_xlabel(r'$\theta_g = \Delta R_{12}/R$')
ax.set_ylabel(r'$(1/N_{\rm jet})\,dN/d\theta_g$')
ax.set_title(r'Groomed opening angle $\theta_g$')

# 2D splitting plane
ax = axes[2]
h = ax.hist2d(theta_g_vals, z_g_vals, bins=30,
              range=[[0, 1], [z_cut, 0.5]], cmap='Blues')
fig.colorbar(h[3], ax=ax, label='jets')
ax.set_xlabel(r'$\theta_g$')
ax.set_ylabel(r'$z_g$')
ax.set_title(r'Splitting plane $(z_g,\,\theta_g)$')

plt.tight_layout()
plt.savefig('demo_softdrop_splitting.png', dpi=150)
plt.show()
print('Saved demo_softdrop_splitting.png')